**Data prep.** Checks which DenseGait pose files (`poses/0.npy` ... `poses/217952.npy`) are actually present and writes `annotations_present.csv` / `annotations_missing.csv`.

In the author's copy 175,247 of 217,953 files were present (42,706 missing). Training uses `annotations_present.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np

# Directory path
poses_dir = "/content/drive/MyDrive/Gait Project/DenseGait/poses"

# Expected file range
expected_start = 0
expected_end = 217952

# Get list of all .npy files in directory
try:
    all_files = os.listdir(poses_dir)
    npy_files = [f for f in all_files if f.endswith('.npy')]

    # Extract file numbers
    present_numbers = []
    for f in npy_files:
        try:
            num = int(f.replace('.npy', ''))
            present_numbers.append(num)
        except ValueError:
            print(f"Warning: Skipping non-numeric file: {f}")

    present_numbers = sorted(present_numbers)

    # Create set of expected numbers
    expected_set = set(range(expected_start, expected_end + 1))
    present_set = set(present_numbers)

    # Find missing files
    missing_numbers = sorted(expected_set - present_set)

    # Find extra files (outside expected range)
    extra_numbers = sorted(present_set - expected_set)

    # Print summary
    print("="*60)
    print("FILE ANALYSIS SUMMARY")
    print("="*60)
    print(f"Expected range: {expected_start} to {expected_end}")
    print(f"Total expected files: {len(expected_set):,}")
    print(f"\nPresent files: {len(present_set):,}")
    print(f"Missing files: {len(missing_numbers):,}")
    print(f"Extra files (outside range): {len(extra_numbers)}")
    print("="*60)

    # Show first and last few missing files
    if missing_numbers:
        print("\nFirst 20 missing files:")
        print(missing_numbers[:20])
        if len(missing_numbers) > 20:
            print(f"\nLast 20 missing files:")
            print(missing_numbers[-20:])

        # Save missing numbers to array
        missing_array = np.array(missing_numbers)
        print(f"\nMissing files saved to 'missing_array' variable")
        print(f"Shape: {missing_array.shape}")
    else:
        print("\n✓ All files are present!")
        missing_array = np.array([])

    # Show extra files if any
    if extra_numbers:
        print(f"\nExtra files (outside expected range):")
        print(extra_numbers[:50])  # Show up to 50
        extra_array = np.array(extra_numbers)
    else:
        extra_array = np.array([])

    # Present files array
    present_array = np.array(present_numbers)

    # Optional: Save to files
    save_to_file = True
    if save_to_file and len(missing_numbers) > 0:
        np.save('/content/missing_files.npy', missing_array)
        with open('/content/missing_files.txt', 'w') as f:
            for num in missing_numbers:
                f.write(f"{num}.npy\n")
        print(f"\n✓ Missing file list saved to:")
        print("  - /content/missing_files.npy")
        print("  - /content/missing_files.txt")

    if save_to_file and len(present_numbers) > 0:
        np.save('/content/present_files.npy', present_array)
        print(f"✓ Present file list saved to /content/present_files.npy")

except FileNotFoundError:
    print(f"Error: Directory not found: {poses_dir}")
    print("Make sure your Google Drive is mounted!")
    print("\nTo mount Drive, run:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
import os
import numpy as np
import pandas as pd

# Directory paths
poses_dir = "/content/drive/MyDrive/Gait Project/DenseGait/poses"
annotations_file = "/content/drive/MyDrive/Gait Project/DenseGait/annotations.csv"

# Expected file range
expected_start = 0
expected_end = 217952

print("="*60)
print("STEP 1: ANALYZING POSES DIRECTORY")
print("="*60)

# Get list of all .npy files in directory
try:
    all_files = os.listdir(poses_dir)
    npy_files = [f for f in all_files if f.endswith('.npy')]

    # Extract file numbers
    present_numbers = []
    for f in npy_files:
        try:
            num = int(f.replace('.npy', ''))
            present_numbers.append(num)
        except ValueError:
            print(f"Warning: Skipping non-numeric file: {f}")

    present_numbers = sorted(present_numbers)

    # Create set of expected numbers
    expected_set = set(range(expected_start, expected_end + 1))
    present_set = set(present_numbers)

    # Find missing files
    missing_numbers = sorted(expected_set - present_set)

    # Print summary
    print(f"Expected range: {expected_start} to {expected_end}")
    print(f"Total expected files: {len(expected_set):,}")
    print(f"\nPresent files: {len(present_set):,}")
    print(f"Missing files: {len(missing_numbers):,}")
    print("="*60)

    print("\n" + "="*60)
    print("STEP 2: PROCESSING ANNOTATIONS FILE")
    print("="*60)

    # Read annotations CSV
    df = pd.read_csv(annotations_file)
    print(f"Total records in annotations.csv: {len(df):,}")

    # Extract file number from file_name column
    df['file_number'] = df['file_name'].str.replace('.npy', '').astype(int)

    # Keep only Female and file_name columns (plus file_number for filtering)
    df_filtered = df[['Female', 'file_name', 'file_number']].copy()

    # Split into present and missing
    df_present = df_filtered[df_filtered['file_number'].isin(present_set)].copy()
    df_missing = df_filtered[df_filtered['file_number'].isin(missing_numbers)].copy()

    # Drop the file_number helper column
    df_present = df_present[['Female', 'file_name']]
    df_missing = df_missing[['Female', 'file_name']]

    print(f"\nRecords with PRESENT pose files: {len(df_present):,}")
    print(f"Records with MISSING pose files: {len(df_missing):,}")

    # Save to CSV files
    present_csv = "/content/drive/MyDrive/Gait Project/DenseGait/annotations_present.csv"
    missing_csv = "/content/drive/MyDrive/Gait Project/DenseGait/annotations_missing.csv"

    df_present.to_csv(present_csv, index=False)
    df_missing.to_csv(missing_csv, index=False)

    print("\n" + "="*60)
    print("FILES SAVED SUCCESSFULLY")
    print("="*60)
    print(f"✓ Present annotations: {present_csv}")
    print(f"  Columns: Female, file_name")
    print(f"  Records: {len(df_present):,}")

    print(f"\n✓ Missing annotations: {missing_csv}")
    print(f"  Columns: Female, file_name")
    print(f"  Records: {len(df_missing):,}")

    # Show sample of each
    print("\n" + "="*60)
    print("SAMPLE DATA - PRESENT FILES (First 5 records)")
    print("="*60)
    print(df_present.head())

    print("\n" + "="*60)
    print("SAMPLE DATA - MISSING FILES (First 5 records)")
    print("="*60)
    print(df_missing.head())

    # Additional statistics
    print("\n" + "="*60)
    print("GENDER DISTRIBUTION")
    print("="*60)
    print(f"\nPresent files:")
    print(f"  Female (≥0.5): {(df_present['Female'] >= 0.5).sum():,}")
    print(f"  Male (<0.5): {(df_present['Female'] < 0.5).sum():,}")

    print(f"\nMissing files:")
    print(f"  Female (≥0.5): {(df_missing['Female'] >= 0.5).sum():,}")
    print(f"  Male (<0.5): {(df_missing['Female'] < 0.5).sum():,}")

    # Save numpy arrays for convenience
    np.save('/content/missing_file_numbers.npy', np.array(missing_numbers))
    np.save('/content/present_file_numbers.npy', np.array(present_numbers))
    print("\n✓ Numpy arrays saved:")
    print("  - /content/missing_file_numbers.npy")
    print("  - /content/present_file_numbers.npy")

except FileNotFoundError as e:
    print(f"Error: File or directory not found: {e}")
    print("\nMake sure:")
    print("1. Your Google Drive is mounted")
    print("2. The paths are correct")
    print("\nTo mount Drive, run:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()